In [ ]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

load_dotenv("../.env")

genai.configure(
    api_key=os.getenv("GOOGLE_API_KEY")
)

model = genai.GenerativeModel(
    "models/gemini-flash-latest"
)


In [ ]:
api_key=os.getenv("GOOGLE_API_KEY")


In [ ]:
import pandas as pd
df= pd.read_csv("../data/stories.csv")
df.head()


In [ ]:
def adapt_story(story):

    adapted_story = story

    for original, replacement in cultural_mapping.items():

        adapted_story = adapted_story.replace(
            original,
            replacement
        )

    return adapted_story


In [ ]:
cultural_mapping = {

    # Names
    "Tom": "Rohan",
    "Jack": "Arjun",
    "Emily": "Ananya",

    # Locations
    "New York": "Delhi",
    "London": "Mumbai",

    # Festivals
    "Christmas": "Diwali",
    "Thanksgiving": "Pongal",

    # Roles
    "king": "Maharaja",
    "queen": "Maharani",
    "knight": "warrior",

    # Places
    "castle": "Raj Mahal"
}


In [ ]:
df["adapted_story"]= df["story_text"].apply(adapt_story)
df.head()


In [ ]:
df.to_csv("../outputs/adapted_stories.csv", index=False)
print("Saved Successfully!")


In [ ]:
for _, row in df.head(3).iterrows():

    print("Original:")
    print(row["story_text"])

    print("\nAdapted:")
    print(row["adapted_story"])

    print("\n" + "="*50)


In [ ]:
story = """
Tom celebrated Christmas in New York.
"""

rule_output = adapt_story(story)
print(rule_output)


In [ ]:
print(model)


In [ ]:
response = model.generate_content("Hello")

print(response.text)


In [ ]:
def llm_adapt_story(story):
    prompt = f"""
    Convert the following western story into
    Indian cultural setting.

    Rules:

    1. Preserve storyline
    2. Preserve moral
    3. Replace names
    4. Replace festivals
    5. Replace places
    6. Make story natural

    Story:
    {story}
"""
    response =model.generate_content(prompt)

    return response.text


In [ ]:
print("RULE BASED")
print(rule_output)

print("\n")

print("LLM BASED")
print(llm_adapt_story(story))


In [ ]:
comparison = {
    "Rule Based":[
        "Fast",
        "Cheap",
        "Limited Context"
    ],

    "LLM Based":[
        "Context Aware",
        "Natural Output",
        "Better Adaptation"
    ]
}

comparison


In [ ]:
import json
with open("../data/cultural_knowledge.json") as f:
    knowledge = json.load(f)
knowledge    


In [ ]:
with open("../data/cultural_knowledge.json","r") as f:
    print(f.read())


In [ ]:
import json

with open("../data/cultural_knowledge.json", "r") as f:
    content = f.read()

print(content)


In [ ]:
import json

with open("../data/cultural_knowledge.json", "r") as f:
    knowledge = json.loads(f.read())

print(knowledge)



In [ ]:
def retrieve_mapping(entity):

    for category in knowledge.values():

        if entity in category:
            return category[entity]

    return "No Mapping Found"


In [ ]:
print(retrieve_mapping("Christmas"))
print(retrieve_mapping("Tom"))
print(retrieve_mapping("New York"))


In [ ]:
def extract_entities(text):

    doc = nlp(text)

    entities = []

    for ent in doc.ents:

        entities.append({
            "entity": ent.text,
            "label": ent.label_
        })

    return entities


In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")


In [ ]:
story = "Tom celebrated Christmas in New York."

entities = extract_entities(story)

print(entities)


In [ ]:
story = "Tom celebrated Christmas in New York."

entities = extract_entities(story)

for item in entities:

    print(
        item["entity"],
        "=>",
        retrieve_mapping(item["entity"])
    )


 # Objective

 ## Build a retrieval Augmented Story Adaptation Pipeline

 ### Context Builder

In [ ]:
def build_context(story):
    entities = extract_entities(story)
    context = []

    for item in entities:
        entity = item["entity"]
        mapping = retrieve_mapping(entity)
        if mapping != "No mapping Found":
            context.append(
                f"{entity} should be adapted as {mapping}"
            )
    return "\n".join(context)        


 ### Test

In [ ]:
story = """
Tom celebrated Christmas with his family in New York.
"""

print(build_context(story))



 ### RAG Prompt

In [ ]:
def rag_adapt_story(story):
    
    context = build_context(story)

    prompt = f"""
    You are a cultural adaptation specialist.

    Context:

    {context}

    Rules:

    1. Preserve storyline
    2. Preserve moral lesson
    3. Use context mappings
    4. Make adaptation natural

    Story:

    {story}
    """

    response = model.generate_content(
        prompt
    )

    return response.text


 ### Test

In [ ]:
story = """
Tom celebrated Christmas with his family in New York.
"""

print(
    rag_adapt_story(story)
)


 ### Compare

In [ ]:
story = """
Tom celebrated Christmas in New York.
"""


 #### Rule Based

In [ ]:
adapt_story(story)


 #### LLM

In [ ]:
llm_adapt_story(story)


 #### RAG

In [ ]:
rag_adapt_story(story)


 ### Evaluation Of Cultural Adaptation Using Semantic Similiarity Metrics

In [ ]:
from sentence_transformers import SentenceTransformer

model_eval = SentenceTransformer(
    "all-MiniLM-L6-v2"
)


 #### Original vs Adapted Story

In [ ]:
original = """
Tom celebrated Christmas with his family in New York.
"""

adapted = """
Rohan celebrated Diwali with his family in Delhi.
"""


 #### Embeddings

In [ ]:
emb1 = model_eval.encode(original)

emb2 = model_eval.encode(adapted)


 #### Similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

score = cosine_similarity(
    [emb1],
    [emb2]
)

print(score)


In [ ]:
def similarity_score(text1, text2):

    emb1 = model_eval.encode(text1)

    emb2 = model_eval.encode(text2)

    score = cosine_similarity(
        [emb1],
        [emb2]
    )

    return score[0][0]


In [ ]:
similarity_score(
    original,
    adapted
)


 #### Dataset Evaluatioj

In [ ]:
df["similarity_score"] = df.apply(
    lambda row: similarity_score(
        row["story_text"],
        row["adapted_story"]
    ),
    axis=1
)


 #### Average Score

In [ ]:
print(
    df["similarity_score"].mean()
)


